In [2]:
!pip install -qU FlagEmbedding langchain-text-splitters numpy scikit-learn

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Nosso "Documento" simulado (Pode ser o texto extraído de um PDF depois)
texto_base = """
O Trabalho de Conclusão de Curso (TCC) é uma etapa fundamental na formação acadêmica. 
Ele permite ao aluno aplicar os conhecimentos adquiridos durante a graduação.
No contexto de Inteligência Artificial, o RAG (Retrieval-Augmented Generation) surgiu 
como uma técnica para mitigar as alucinações de modelos de linguagem (LLMs).
O BGE-M3 é um modelo de embeddings multilíngue desenvolvido pela BAAI.
Ele se destaca por suportar textos longos de até 8192 tokens e gerar vetores densos, 
esparsos e colbert simultaneamente.
Para avaliar um sistema RAG, métricas como precisão de recuperação (Hit Rate) 
e relevância (MRR - Mean Reciprocal Rank) são frequentemente utilizadas.
"""

# 2. Configurando o Picador
# chunk_size = tamanho máximo de caracteres por pedaço
# chunk_overlap = quantos caracteres o chunk 2 "rouba" do final do chunk 1 para não perder contexto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, 
    chunk_overlap=30,
    separators=["\n\n", "\n", ".", " ", ""] # Tenta cortar no parágrafo, depois no ponto, depois no espaço...
)

chunks = text_splitter.split_text(texto_base)

print(f"O documento foi dividido em {len(chunks)} chunks:\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(chunk)

/home/ryan-mazzeu/Desktop/TCC/TCC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


O documento foi dividido em 7 chunks:

--- Chunk 0 ---
O Trabalho de Conclusão de Curso (TCC) é uma etapa fundamental na formação acadêmica.
--- Chunk 1 ---
Ele permite ao aluno aplicar os conhecimentos adquiridos durante a graduação.
--- Chunk 2 ---
No contexto de Inteligência Artificial, o RAG (Retrieval-Augmented Generation) surgiu
--- Chunk 3 ---
como uma técnica para mitigar as alucinações de modelos de linguagem (LLMs).
O BGE-M3 é um modelo de embeddings multilíngue desenvolvido pela BAAI.
--- Chunk 4 ---
Ele se destaca por suportar textos longos de até 8192 tokens e gerar vetores densos, 
esparsos e colbert simultaneamente.
--- Chunk 5 ---
Para avaliar um sistema RAG, métricas como precisão de recuperação (Hit Rate)
--- Chunk 6 ---
e relevância (MRR - Mean Reciprocal Rank) são frequentemente utilizadas.


In [4]:
from FlagEmbedding import BGEM3FlagModel

print("Carregando o modelo BGE-M3... Isso pode levar alguns segundos na primeira vez.")

# use_fp16=True ajuda a economizar muita memória de vídeo na GPU sem perder precisão
modelo_bge = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) 

print("Modelo carregado com sucesso!")

Carregando o modelo BGE-M3... Isso pode levar alguns segundos na primeira vez.


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 334.58it/s]

Modelo carregado com sucesso!


In [5]:
import numpy as np

print("Gerando embeddings para os chunks...")

# O método encode transforma o texto em números.
saida_bge = modelo_bge.encode(
    chunks, 
    batch_size=12, 
    max_length=8192, 
    return_dense=True, 
    return_sparse=False, 
    return_colbert_vecs=False
)

# Extraímos apenas os vetores densos (uma matriz de números)
vetores_documentos = saida_bge['dense_vecs']

print(f"Pronto! Temos {vetores_documentos.shape[0]} vetores.")
print(f"Cada vetor tem {vetores_documentos.shape[1]} dimensões matemáticas.")

Gerando embeddings para os chunks...
Pronto! Temos 7 vetores.
Cada vetor tem 1024 dimensões matemáticas.


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. A pergunta do usuário
query = "Quais métricas eu posso usar para avaliar meu sistema?"
print(f"Query do usuário: '{query}'\n")

# 2. Transformamos a pergunta em números
# CORREÇÃO AQUI: Colocamos a query dentro de uma lista [query]
vetor_query = modelo_bge.encode([query], return_dense=True)['dense_vecs']

# 3. Calculamos a Similaridade de Cosseno 
similaridades = cosine_similarity(vetor_query, vetores_documentos)[0]

# 4. Pegamos os Top K mais relevantes (ex: os 2 melhores)
k = 2
indices_top_k = np.argsort(similaridades)[::-1][:k]

# 5. Imprimimos o resultado!
print(f"--- TOP {k} TRECHOS MAIS RELEVANTES RECUPERADOS ---")
for posicao, indice in enumerate(indices_top_k):
    score = similaridades[indice]
    print(f"\n[{posicao + 1}º Lugar] (Score de similaridade: {score:.4f})")
    print(f"Texto: {chunks[indice]}")

Query do usuário: 'Quais métricas eu posso usar para avaliar meu sistema?'

--- TOP 2 TRECHOS MAIS RELEVANTES RECUPERADOS ---

[1º Lugar] (Score de similaridade: 0.6338)
Texto: Para avaliar um sistema RAG, métricas como precisão de recuperação (Hit Rate)

[2º Lugar] (Score de similaridade: 0.4541)
Texto: e relevância (MRR - Mean Reciprocal Rank) são frequentemente utilizadas.
